# 15 — Qubit State Tomography

Consolidates legacy experiments 36, 37, and 38.

1. **Single-state tomography** — reconstruct one qubit state on the Bloch sphere (Exp 36)
2. **Multi-state tomography** — sweep through multiple preparation states (Exp 37)
3. **Storage Wigner tomography** — full Wigner function of the storage cavity (Exp 38)

## 1. Shared Session Bootstrap

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from qualang_tools.units import unit

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "qubox").exists():
    REPO_ROOT = Path(r"E:\qubox")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qubox.notebook import (
    QubitStateTomography,
    StorageWignerTomography,
    load_stage_checkpoint,
    open_notebook_stage,
    save_stage_checkpoint,
)

REGISTRY_BASE = Path(r"E:\qubox")
SAMPLE_ID = "post_cavity_sample_A"
COOLDOWN_ID = "cd_2025_02_22"
QOP_IP = "10.157.36.68"
CLUSTER_NAME = "Cluster_2"

stage = open_notebook_stage(
    stage_name="15_qubit_state_tomography",
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    qop_ip=QOP_IP,
    cluster_name=CLUSTER_NAME,
    force_reopen=True,
    close_existing=True,
)
session = stage.session
attr = stage.attr
SESSION_BOOTSTRAP_PATH = stage.bootstrap_path
u = unit(coerce_to_integer=True)

gate_checkpoint = load_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="14_gate_calibration_benchmarking",
)

print(f"Repo root on sys.path: {REPO_ROOT}")
print(f"Shared session bootstrap: {SESSION_BOOTSTRAP_PATH}")
print(f"Stage checkpoint path: {stage.checkpoint_path}")
print(f"QM endpoint: {QOP_IP} ({CLUSTER_NAME})")
if gate_checkpoint is not None:
    print(f"Prior stage 14 status: {gate_checkpoint['status']}")

## 2. Tomography Defaults

In [ ]:
# --- Single-state tomography (Exp 36) ---
TOMO_1_STATE_PREP = "x90"  # Prepare |+⟩ state
TOMO_1_N_AVG = 10000

# --- Multi-state tomography (Exp 37) ---
TOMO_MULTI_STATES = ["I", "x90", "x180", "y90", "y180"]
TOMO_MULTI_N_AVG = 10000

# --- Storage Wigner tomography (Exp 38) ---
WIGNER_ALPHA_MAX = 3.0
WIGNER_ALPHA_STEP = 0.1
WIGNER_N_AVG = 10000

print("Tomography settings:")
print(f"  Single-state prep: {TOMO_1_STATE_PREP}, n_avg={TOMO_1_N_AVG}")
print(f"  Multi-state preps: {TOMO_MULTI_STATES}, n_avg={TOMO_MULTI_N_AVG}")
print(f"  Wigner: α_max={WIGNER_ALPHA_MAX}, dα={WIGNER_ALPHA_STEP}, n_avg={WIGNER_N_AVG}")

## 3. Single-State Qubit Tomography — Exp 36

Reconstruct one qubit state via X, Y, Z measurement bases.

In [ ]:
tomo_single = QubitStateTomography(session)
tomo_single_result = tomo_single.run(
    state_prep=TOMO_1_STATE_PREP,
    n_avg=TOMO_1_N_AVG,
)
tomo_single_analysis = tomo_single.analyze(tomo_single_result, update_calibration=False)
tomo_single.plot(tomo_single_analysis)

print("Single-state tomography complete.")
if hasattr(tomo_single_analysis, "metrics"):
    for k, v in tomo_single_analysis.metrics.items():
        print(f"  {k}: {v}")

## 4. Multi-State Qubit Tomography — Exp 37

Sweep through several preparation states and verify Bloch sphere coordinates.

In [ ]:
tomo_results = {}

for state_op in TOMO_MULTI_STATES:
    tomo_multi = QubitStateTomography(session)
    result = tomo_multi.run(
        state_prep=state_op,
        n_avg=TOMO_MULTI_N_AVG,
    )
    analysis = tomo_multi.analyze(result, update_calibration=False)
    tomo_multi.plot(analysis)
    tomo_results[state_op] = analysis
    print(f"  {state_op}: done")

print("Multi-state tomography sweep complete.")

## 5. Storage Wigner Tomography — Exp 38

Full Wigner function measurement of the storage cavity.

In [ ]:
wigner = StorageWignerTomography(session)
wigner_result = wigner.run(
    alpha_max=WIGNER_ALPHA_MAX,
    alpha_step=WIGNER_ALPHA_STEP,
    n_avg=WIGNER_N_AVG,
)
wigner_analysis = wigner.analyze(wigner_result, update_calibration=False)
wigner.plot(wigner_analysis)

print("Storage Wigner tomography complete.")

## 6. Save Checkpoint

In [ ]:
stage_checkpoint_path = save_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="15_qubit_state_tomography",
    status="characterized",
    summary="Qubit single/multi-state tomography and storage Wigner function measured.",
    consumed_inputs={
        "tomo_single_state_prep": TOMO_1_STATE_PREP,
        "tomo_single_n_avg": TOMO_1_N_AVG,
        "tomo_multi_states": TOMO_MULTI_STATES,
        "wigner_alpha_max": WIGNER_ALPHA_MAX,
        "wigner_alpha_step": WIGNER_ALPHA_STEP,
        "wigner_n_avg": WIGNER_N_AVG,
    },
    persisted_outputs={},
    advisory_outputs={},
    next_stage="16_readout_calibration",
    notes=[
        "All tomography experiments are characterization-only — no calibrations applied.",
        "Wigner function quality depends on displacement ops registered in notebook 08.",
    ],
    metrics={},
)

print(f"Stage checkpoint saved to: {stage_checkpoint_path}")